In [ ]:
#mental model is like this

#x_0 is a starting distribution (gaussian noise to easily sample from)

#x_1 is a end (target distribution) -> t=0 is starting distribution, t=1 is end distribution

#interpolation is x_t=(1-t)*x_0 + x_1*t

#derivative of the interpolant is a velocity field (how everything in v_t changes per time)

#vector field is then d/dt[x_t]=d/dt[(1-t)*x_0+x_1*t]=x_1-x_0

#flow matching neural network parametrizes v(parameters, t)->approximates [x_1-x_0]

#loss is then Loss=[v(parameters, t) - [x_1-x_0]]

In [29]:
import torch
from torch.distributions import Normal

class ShiftedScaledGaussian1D:
  def __init__(self, scaling=0.5, shifting=2.0, device="cpu"):
    self.scaling=torch.tensor(scaling, dtype=torch.float32, device=device)
    self.shifting=torch.tensor(shifting, dtype=torch.float32, device=device)

    self._dist=Normal(loc=self.shifting, scale=self.scaling)

  def sample(self, num_samples):
    samples=self._dist.sample((num_samples, 1))
    return samples.to(self.shifting.device)

  def pdf(self, x):
    return torch.exp(self._dist.log_prob(x))

  def log_prob(self, x):
    return self._dist.log_prob(x)

In [30]:
base_distribution=torch.distributions.Normal(0.0, 1.0)

target_distribution=ShiftedScaledGaussian1D(scaling=0.5, shifting=2.0)

target_samples=target_distribution.sample(1000)

base_samples=base_distribution.sample((1000,1))



In [31]:
def exact_velocity(x, t, target):
  u=target.shifting
  s=target.scaling

  velocity_vector_field= u + ((t*(s**2)-(1-t))/((1-t)**2+(t**2)*(s**2)))*(x-(t*u))
  return velocity_vector_field


In [32]:
exact_velocity(0.5, 0.0, target_distribution)

tensor(1.5000)

In [38]:
oracle = lambda x,t: exact_velocity(x, t, target_distribution)
def euler_sample(velocity_field, x0, steps):
  time_steps=1/steps
  stack=[x0]

  with torch.no_grad():
    for i in range(steps):
      t = torch.full_like(x0, time_steps * i)
      velocity = velocity_field(x0, t)
      x=x0+time_steps*velocity
      x0=x
      stack.append(x)
  return torch.stack(stack)



In [41]:
traj = euler_sample(oracle, torch.tensor([[1.]]), 2)  # inside, velocity_field is oracle
print(traj)

tensor([[[1.0000]],

        [[1.5000]],

        [[2.2000]]])


In [47]:
import torch.nn as nn
class VelocityMLP(nn.Module):
  def __init__(self):
    super().__init__()
    self.linear1=nn.Linear(2, 256)
    self.silu=nn.SiLU()
    self.linear2=nn.Linear(256, 256)
    self.silu2=nn.SiLU()
    self.linear3=nn.Linear(256, 256)
    self.silu3=nn.SiLU()
    self.linear4=nn.Linear(256, 1)
  def forward(self, x, t):
    input=torch.cat([x,t],dim=-1)
    x=self.linear1(input)
    x=self.silu(x)
    x=self.linear2(x)
    x=self.silu2(x)
    x=self.linear3(x)
    x=self.silu3(x)
    x=self.linear4(x)
    return x



In [50]:
model=VelocityMLP()
model.train()

VelocityMLP(
  (linear1): Linear(in_features=2, out_features=256, bias=True)
  (silu): SiLU()
  (linear2): Linear(in_features=256, out_features=256, bias=True)
  (silu2): SiLU()
  (linear3): Linear(in_features=256, out_features=256, bias=True)
  (silu3): SiLU()
  (linear4): Linear(in_features=256, out_features=1, bias=True)
)

In [52]:
oracle=lambda x,t: exact_velocity(x, t, target_distribution)
def fm_loss(velocity_field, x_1):
  x_0=torch.randn_like(x_1)
  time=torch.rand(x_1.shape[0], 1, device=x_1.device)
  #interpolation equation
  x_t=(1-time)*x_0+x_1*time
  velocity_field_estimate=velocity_field(x_t, time)

  loss=torch.mean((x_1-x_0-velocity_field_estimate).pow(2))
  return loss


In [53]:
x1 = target_distribution.sample(100_000)

In [55]:
with torch.no_grad():
  print(fm_loss(model, x1))
  print(fm_loss(oracle, x1))

tensor(5.1030)
tensor(0.7866)


In [57]:
#training loops
model=VelocityMLP()
optimizer=torch.optim.Adam(model.parameters(), lr=1e-3)
batch_size=1024
steps=5000

for i in range(steps):
  x1=target_distribution.sample(batch_size)
  loss=fm_loss(model, x1)

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  print("loss is: ", loss.item())


loss is:  5.18485164642334
loss is:  4.810818195343018
loss is:  4.128026485443115
loss is:  3.662064552307129
loss is:  3.08518123626709
loss is:  2.4316203594207764
loss is:  2.1133534908294678
loss is:  2.067298173904419
loss is:  2.246152639389038
loss is:  2.2063817977905273
loss is:  1.830406904220581
loss is:  1.6124207973480225
loss is:  1.4014374017715454
loss is:  1.273547649383545
loss is:  1.1853699684143066
loss is:  1.201723337173462
loss is:  1.10318124294281
loss is:  1.2666524648666382
loss is:  1.049242615699768
loss is:  1.1087534427642822
loss is:  0.9993066787719727
loss is:  0.9163879156112671
loss is:  0.9966828227043152
loss is:  0.9844167232513428
loss is:  0.984272837638855
loss is:  1.0658714771270752
loss is:  1.0528031587600708
loss is:  1.0072104930877686
loss is:  1.0851900577545166
loss is:  1.0211137533187866
loss is:  1.0013861656188965
loss is:  0.9458326697349548
loss is:  1.012131690979004
loss is:  0.9521545171737671
loss is:  0.9263578653335571
lo

In [58]:
traj = euler_sample(model, torch.tensor([[1.]]), 2)  # inside, velocity_field is oracle
print(traj)

tensor([[[1.0000]],

        [[1.4973]],

        [[2.1696]]])
